# Data Preparation – Semantische Analyse

In [ ]:
import re
import numpy as np
import pandas as pd
import spacy

# python -m spacy download de_core_news_sm
nlp = spacy.load("de_core_news_sm")

## 1. Daten laden

In [ ]:
INPUT_PATH  = "./data/transformed/digitalhaushalt_transformed_with_titel_text.csv"
OUTPUT_PATH = "./data/transformed/digitalhaushalt_semantic_features.csv"

df = pd.read_csv(INPUT_PATH)
print(f"Zeilen: {len(df):,}  |  Spalten: {df.shape[1]}")
df.head(3)

## 2. Text bereinigen

In [ ]:
NOISE_MARKERS = [
    r"\.{4,}",          # ......... (Tabulator-Ersatz)
    r"Haushaltsvermerk",
    # r"Erläuterungen",
]

def clean_titel(text: str) -> str:
    text = str(text)
    for marker in NOISE_MARKERS:
        m = re.search(marker, text)
        if m:
            text = text[:m.start()]
    # Whitespace normalisieren
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["cleaned_text"] = df["titel_text"].apply(clean_titel)

# Sanity check: wie viel kürzer ist cleaned_text im Schnitt?
df["_ratio"] = df["cleaned_text"].str.len() / df["titel_text"].astype(str).str.len().replace(0, np.nan)
print(f"Durchschn. Längen-Ratio cleaned/original: {df['_ratio'].mean():.2%}")
df.drop(columns=["_ratio"], inplace=True)

## 3. Text-Features

In [ ]:
# ── 1. CLEANING LOGIK ─────────────────────────────────────────────────────────
NOISE_MARKERS = [
    r"\.{4,}",
    r"Haushaltsvermerk",
]

def clean_titel_standard(text: str) -> str:
    text = str(text)
    for marker in NOISE_MARKERS:
        m = re.search(marker, text)
        if m:
            text = text[:m.start()]
    return re.sub(r"\s+", " ", text).strip()


def clean_titel_fully(text: str) -> str:
    text = str(text)
    m = re.search(r"Erläuterungen", text)
    if m:
        text = text[:m.start()]
    return re.sub(r"\s+", " ", text).strip()


# ── 2. TEXT-STUFEN ERSTELLEN ──────────────────────────────────────────────────
df["text_raw"] = df["titel_text"].fillna("").astype(str)
df["text_clean"] = df["text_raw"].apply(clean_titel_standard)
df["text_fully_clean"] = df["text_clean"].apply(clean_titel_fully)


# ── 3. NLP FEATURE EXTRAKTION ─────────────────────────────────────────────────
ACTION_WORDS = {
    # --- Geldflüsse & Zuweisungen ---
    "ausgaben", "kosten", "finanzierung", "zuschuss", "zuschüsse", 
    "förderung", "mittel", "zuwendung", "zuweisungen", "beitrag",
    "fördern", "bezuschussen", "finanzieren",

    # --- Beschaffung & Realisierung ---
    "beschaffung", "erwerb", "kauf", "anmietung", "bereitstellung",
    "beschaffen", "erwerben", "kaufen", "bereitstellen",

    # --- Betrieb & Infrastruktur ---
    "betrieb", "wartung", "unterhalt", "pflege", "administration",
    "betreiben", "warten", "unterhalten",

    # --- Innovation & Erstellung ---
    "entwicklung", "investition", "forschung", "aufbau", "modernisierung",
    "erstellung", "konzeption", "umsetzung", "einführung",
    "entwickeln", "investieren", "aufbauen", "modernisieren", "umsetzen"
}

def extract_text_features(text: str) -> dict:
    doc = nlp(str(text))
    tokens = [t for t in doc if not t.is_space]
    nouns  = [t for t in tokens if t.pos_ == "NOUN"]
    actions = [t for t in tokens if t.lemma_.lower() in ACTION_WORDS]

    n_tokens = len(tokens)
    noun_ratio = len(nouns) / max(n_tokens, 1)
    action_density = len(actions) / max(n_tokens, 1)

    return {
        "token_count"    : n_tokens,
        "noun_ratio"     : round(noun_ratio, 4),
        "action_density" : round(action_density, 4),
    }


# ── 4. PIPELINE LOOP FÜR ALLE 3 TEXTE ─────────────────────────────────────────
for mode in ["raw", "clean", "fully_clean"]:
    features = df[f"text_{mode}"].apply(lambda x: pd.Series(extract_text_features(x)))
    
    df[f"{mode}_token_count"]    = features["token_count"]
    df[f"{mode}_noun_ratio"]     = features["noun_ratio"]
    df[f"{mode}_action_density"] = features["action_density"]

df["has_noise"] = df["text_clean"].str.len() < df["text_raw"].str.len() * 0.95

## 5. Nur digitale Positionen Boolean

In [ ]:
if "digi_soll_weit" in df.columns:
    df["is_digital"] = df["digi_soll_weit"] > 0
    print(f"Digitale Positionen: {df['is_digital'].sum():,} / {len(df):,}")

## 6. Speichern

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Gespeichert: {OUTPUT_PATH}")
print(f"Shape: {df.shape}")